# Week 15: Model Evaluation & Fairness (Bias Detection, Robustness)
# 第 15 週：模型評估與公平性（偏誤檢測、穩健性）

## 學習目標 Learning Objectives
1. 深入理解混淆矩陣，視覺化 TP/FP/TN/FN 在特徵空間中的分布
2. 計算 Precision、Recall、F1、Specificity 等指標隨閾值的變化
3. 繪製多類別 ROC 曲線（One-vs-Rest）和 PR 曲線
4. 閾值優化：找到最佳決策閾值
5. 校準曲線 (Calibration Curve)：評估預測機率的可靠性
6. 公平性指標：Demographic Parity、Equalized Odds、Predictive Parity
7. 資料偏誤：訓練資料不平衡如何產生偏見
8. 偏誤緩解：重新加權和閾值調整
9. 模型穩健性：在噪音/偏移資料上測試

**環境需求 Required packages**：numpy, matplotlib, sklearn, scipy

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.datasets import make_classification

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

print('所有套件匯入成功！ All packages imported successfully!')

In [ ]:
# 建立包含受保護屬性的合成資料集
# Create a synthetic dataset with a protected attribute (group A / B)
np.random.seed(42)
n_total = 2000

# Group A: 較高的正類基礎比率 (base rate = 0.4)
n_A = 1000
X_A = np.random.randn(n_A, 2) + np.array([0.5, 0.3])
y_A = (X_A[:, 0] + X_A[:, 1] + np.random.randn(n_A) * 1.2 > 1.0).astype(int)

# Group B: 較低的正類基礎比率 (base rate = 0.25)
n_B = 1000
X_B = np.random.randn(n_B, 2) + np.array([-0.3, -0.2])
y_B = (X_B[:, 0] + X_B[:, 1] + np.random.randn(n_B) * 1.2 > 1.0).astype(int)

# 合併
X_all = np.vstack([X_A, X_B])
y_all = np.concatenate([y_A, y_B])
group = np.array(['A'] * n_A + ['B'] * n_B)

# 分割訓練/測試
X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    X_all, y_all, group, test_size=0.3, random_state=42, stratify=y_all
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Dataset size: {len(X_all)}')
print(f'Group A: {n_A} samples, positive rate = {y_A.mean():.3f}')
print(f'Group B: {n_B} samples, positive rate = {y_B.mean():.3f}')
print(f'Overall positive rate: {y_all.mean():.3f}')
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

# 訓練一個基線模型
clf_base = LogisticRegression(random_state=42, max_iter=1000)
clf_base.fit(X_train_s, y_train)
y_pred = clf_base.predict(X_test_s)
y_proba = clf_base.predict_proba(X_test_s)[:, 1]

print(f'\nBaseline Logistic Regression Accuracy: {accuracy_score(y_test, y_pred):.4f}')

---
## Part 1: 混淆矩陣深入 Confusion Matrix Deep Dive

混淆矩陣不只是一個 2x2 的表格 —— 我們可以把 TP/FP/TN/FN 映射回**特徵空間**，
直觀地看到模型在哪些區域犯了什麼類型的錯誤。

In [ ]:
# 視覺化 TP/FP/TN/FN 在 2D 特徵空間中的分布
cm = confusion_matrix(y_test, y_pred)

# 將測試樣本分成 TP, FP, TN, FN
tp_mask = (y_test == 1) & (y_pred == 1)
fp_mask = (y_test == 0) & (y_pred == 1)
tn_mask = (y_test == 0) & (y_pred == 0)
fn_mask = (y_test == 1) & (y_pred == 0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 左圖：2D 散點圖標示 TP/FP/TN/FN
ax = axes[0]
ax.scatter(X_test_s[tn_mask, 0], X_test_s[tn_mask, 1], c='royalblue',
           alpha=0.5, s=30, label=f'TN ({tn_mask.sum()})', marker='o')
ax.scatter(X_test_s[tp_mask, 0], X_test_s[tp_mask, 1], c='forestgreen',
           alpha=0.5, s=30, label=f'TP ({tp_mask.sum()})', marker='o')
ax.scatter(X_test_s[fp_mask, 0], X_test_s[fp_mask, 1], c='red',
           alpha=0.7, s=60, label=f'FP ({fp_mask.sum()})', marker='x')
ax.scatter(X_test_s[fn_mask, 0], X_test_s[fn_mask, 1], c='orange',
           alpha=0.7, s=60, label=f'FN ({fn_mask.sum()})', marker='x')

# 繪製決策邊界
xx, yy = np.meshgrid(np.linspace(X_test_s[:, 0].min()-1, X_test_s[:, 0].max()+1, 200),
                      np.linspace(X_test_s[:, 1].min()-1, X_test_s[:, 1].max()+1, 200))
Z = clf_base.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('TP/FP/TN/FN in Feature Space\nTP/FP/TN/FN 在特徵空間中的分布', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 右圖：混淆矩陣熱力圖
ax = axes[1]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Neg (0)', 'Predicted Pos (1)'], fontsize=10)
ax.set_yticklabels(['Actual Neg (0)', 'Actual Pos (1)'], fontsize=10)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix Heatmap\n混淆矩陣熱力圖', fontsize=12)

labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{labels[i][j]}\n{cm[i, j]}',
                ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.colorbar(im, ax=ax, label='Count')
plt.tight_layout()
plt.show()

print('觀察重點 Key Observations:')
print('  FP (紅色x): 模型誤判為正但實際為負 → 決策邊界附近')
print('  FN (橙色x): 模型漏判，實際為正但預測為負 → 也在邊界附近')
print('  大部分錯誤集中在決策邊界的不確定區域')

---
## Part 2: 分類指標隨閾值變化 Classification Metrics vs Threshold

閾值 (Threshold) 決定了模型如何將預測機率轉化為類別標籤。
不同的閾值會改變 Precision、Recall、F1 等指標的平衡。

In [ ]:
# 計算各指標隨閾值的變化 Metrics vs threshold
thresholds = np.linspace(0.01, 0.99, 200)

precisions = []
recalls = []
f1_scores_arr = []
specificities = []
balanced_accs = []

for thresh in thresholds:
    y_pred_t = (y_proba >= thresh).astype(int)
    
    # 避免除以零
    tp = np.sum((y_test == 1) & (y_pred_t == 1))
    fp = np.sum((y_test == 0) & (y_pred_t == 1))
    tn = np.sum((y_test == 0) & (y_pred_t == 0))
    fn = np.sum((y_test == 1) & (y_pred_t == 0))
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    bal_acc = (rec + spec) / 2
    
    precisions.append(prec)
    recalls.append(rec)
    f1_scores_arr.append(f1)
    specificities.append(spec)
    balanced_accs.append(bal_acc)

# 找最佳 F1 閾值
best_f1_idx = np.argmax(f1_scores_arr)
best_threshold = thresholds[best_f1_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：所有指標隨閾值變化
ax = axes[0]
ax.plot(thresholds, precisions, 'b-', linewidth=2, label='Precision')
ax.plot(thresholds, recalls, 'r-', linewidth=2, label='Recall')
ax.plot(thresholds, f1_scores_arr, 'g-', linewidth=2.5, label='F1-Score')
ax.plot(thresholds, specificities, 'purple', linewidth=2, label='Specificity', linestyle='--')
ax.plot(thresholds, balanced_accs, 'orange', linewidth=2, label='Balanced Acc', linestyle='-.')
ax.axvline(x=best_threshold, color='gray', linestyle=':', linewidth=1.5,
           label=f'Best F1 threshold = {best_threshold:.3f}')
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, alpha=0.3, label='Default (0.5)')
ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel('Metric Value', fontsize=11)
ax.set_title('Classification Metrics vs Threshold\n分類指標隨閾值變化', fontsize=12)
ax.legend(fontsize=8, loc='center left')
ax.grid(True, alpha=0.3)

# 右圖：Precision-Recall 權衡
ax = axes[1]
ax.plot(recalls, precisions, 'b-', linewidth=2)
ax.plot(recalls[best_f1_idx], precisions[best_f1_idx], 'ro', markersize=12,
        label=f'Best F1 (P={precisions[best_f1_idx]:.3f}, R={recalls[best_f1_idx]:.3f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Trade-off\nPrecision-Recall 權衡', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f'Best F1 threshold: {best_threshold:.3f}')
print(f'At this threshold:')
print(f'  Precision   = {precisions[best_f1_idx]:.4f}')
print(f'  Recall      = {recalls[best_f1_idx]:.4f}')
print(f'  F1          = {f1_scores_arr[best_f1_idx]:.4f}')
print(f'  Specificity = {specificities[best_f1_idx]:.4f}')

---
## Part 3: 多類別 ROC 和 PR 曲線 Multi-class ROC & PR Curves

多類別問題使用 **One-vs-Rest (OvR)** 策略：
為每個類別分別計算 ROC/PR 曲線，將該類別視為正類，其餘為負類。

In [ ]:
# 生成三類分類資料 Generate 3-class classification data
np.random.seed(42)
X_mc, y_mc = make_classification(
    n_samples=1500, n_features=10, n_informative=6,
    n_classes=3, n_clusters_per_class=1, random_state=42
)
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_mc, y_mc, test_size=0.3, random_state=42, stratify=y_mc
)

scaler_mc = StandardScaler()
X_train_mc_s = scaler_mc.fit_transform(X_train_mc)
X_test_mc_s = scaler_mc.transform(X_test_mc)

# 訓練多類別模型
clf_mc = LogisticRegression(random_state=42, max_iter=1000, multi_class='ovr')
clf_mc.fit(X_train_mc_s, y_train_mc)
y_proba_mc = clf_mc.predict_proba(X_test_mc_s)

# Binarize labels for OvR
y_test_bin = label_binarize(y_test_mc, classes=[0, 1, 2])
n_classes = 3
class_names = ['Class 0', 'Class 1', 'Class 2']
colors_mc = ['#E74C3C', '#3498DB', '#2ECC71']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左圖：ROC 曲線 (One-vs-Rest)
ax = axes[0]
auc_scores = []
for i in range(n_classes):
    fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], y_proba_mc[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_scores.append(auc_i)
    ax.plot(fpr_i, tpr_i, color=colors_mc[i], linewidth=2,
            label=f'{class_names[i]} (AUC = {auc_i:.3f})')

# Micro-average ROC
fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_mc.ravel())
auc_micro = auc(fpr_micro, tpr_micro)
ax.plot(fpr_micro, tpr_micro, color='black', linewidth=2, linestyle='--',
        label=f'Micro-Avg (AUC = {auc_micro:.3f})')

ax.plot([0, 1], [0, 1], 'k:', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Multi-class ROC Curves (One-vs-Rest)\n多類別 ROC 曲線', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# 右圖：PR 曲線 (One-vs-Rest)
ax = axes[1]
for i in range(n_classes):
    prec_i, rec_i, _ = precision_recall_curve(y_test_bin[:, i], y_proba_mc[:, i])
    ap_i = average_precision_score(y_test_bin[:, i], y_proba_mc[:, i])
    ax.plot(rec_i, prec_i, color=colors_mc[i], linewidth=2,
            label=f'{class_names[i]} (AP = {ap_i:.3f})')

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Multi-class PR Curves (One-vs-Rest)\n多類別 PR 曲線', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print('Multi-class AUC scores:')
for i, name in enumerate(class_names):
    print(f'  {name}: AUC = {auc_scores[i]:.4f}')
print(f'  Micro-average AUC: {auc_micro:.4f}')

---
## Part 4: 閾值優化 Threshold Optimization

不同的應用場景需要不同的閾值選擇策略：
- **醫療篩檢**：高 Recall（不漏診）→ 較低閾值
- **垃圾郵件**：高 Precision（不誤判）→ 較高閾值
- **平衡**：最大化 F1 或 Balanced Accuracy

In [ ]:
# 閾值優化視覺化 Threshold optimization visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 三種策略
strategies = [
    ('Maximize F1\n(Balanced)', np.array(f1_scores_arr), 'F1-Score'),
    ('Maximize Recall >= 0.9\n(Medical Screening)', None, 'Recall'),
    ('Maximize Precision >= 0.9\n(Spam Filter)', None, 'Precision'),
]

for idx, (title, _, metric_name) in enumerate(strategies):
    ax = axes[idx]
    ax.plot(thresholds, precisions, 'b-', linewidth=1.5, label='Precision')
    ax.plot(thresholds, recalls, 'r-', linewidth=1.5, label='Recall')
    ax.plot(thresholds, f1_scores_arr, 'g-', linewidth=2, label='F1')
    
    if idx == 0:  # Max F1
        best_idx = np.argmax(f1_scores_arr)
        ax.axvline(x=thresholds[best_idx], color='green', linestyle='--', linewidth=2)
        ax.plot(thresholds[best_idx], f1_scores_arr[best_idx], 'g*', markersize=15)
        ax.text(thresholds[best_idx] + 0.02, 0.95,
                f't={thresholds[best_idx]:.2f}\nF1={f1_scores_arr[best_idx]:.3f}',
                fontsize=9)
    elif idx == 1:  # High Recall
        valid = [i for i, r in enumerate(recalls) if r >= 0.9]
        if valid:
            best_idx = valid[-1]  # 最高閾值滿足 recall >= 0.9
            ax.axvline(x=thresholds[best_idx], color='red', linestyle='--', linewidth=2)
            ax.plot(thresholds[best_idx], recalls[best_idx], 'r*', markersize=15)
            ax.text(thresholds[best_idx] + 0.02, 0.85,
                    f't={thresholds[best_idx]:.2f}\nR={recalls[best_idx]:.3f}\nP={precisions[best_idx]:.3f}',
                    fontsize=9)
    else:  # High Precision
        valid = [i for i, p in enumerate(precisions) if p >= 0.9]
        if valid:
            best_idx = valid[0]  # 最低閾值滿足 precision >= 0.9
            ax.axvline(x=thresholds[best_idx], color='blue', linestyle='--', linewidth=2)
            ax.plot(thresholds[best_idx], precisions[best_idx], 'b*', markersize=15)
            ax.text(thresholds[best_idx] + 0.02, 0.85,
                    f't={thresholds[best_idx]:.2f}\nP={precisions[best_idx]:.3f}\nR={recalls[best_idx]:.3f}',
                    fontsize=9)
    
    ax.set_xlabel('Threshold', fontsize=10)
    ax.set_ylabel('Metric Value', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Threshold Optimization Strategies\n閾值優化策略',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('閾值選擇取決於業務場景 Threshold depends on business context:')
print('  醫療篩檢 → 低閾值，寧可誤判也不漏診 (High Recall)')
print('  垃圾郵件 → 高閾值，寧可漏判也不誤判 (High Precision)')
print('  一般平衡 → 最大化 F1 (Balanced)')

---
## Part 5: 校準曲線 Calibration Curves

模型校準衡量：**當模型預測機率為 p 時，事件實際發生的頻率是否也是 p？**

完美校準的模型在校準圖上會落在 $y = x$ 對角線上。

In [ ]:
# 比較不同模型的校準曲線
# Train multiple models for calibration comparison
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (Platt)': SVC(probability=True, random_state=42),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_cal = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

for (name, model), color in zip(models.items(), colors_cal):
    model.fit(X_train_s, y_train)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test_s)[:, 1]
    else:
        proba = model.decision_function(X_test_s)
    
    # 校準曲線
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, proba, n_bins=10, strategy='uniform'
    )
    
    # Brier Score
    brier = brier_score_loss(y_test, proba)
    
    axes[0].plot(mean_predicted_value, fraction_of_positives, 's-',
                 color=color, linewidth=2, markersize=6,
                 label=f'{name} (Brier={brier:.4f})')
    
    # 預測機率分布直方圖
    axes[1].hist(proba, bins=20, alpha=0.4, color=color,
                 label=name, density=True)

# 對角線（完美校準）
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfectly Calibrated')
axes[0].set_xlabel('Mean Predicted Probability', fontsize=11)
axes[0].set_ylabel('Fraction of Positives', fontsize=11)
axes[0].set_title('Calibration Curve (Reliability Diagram)\n校準曲線', fontsize=12)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].set_aspect('equal')

axes[1].set_xlabel('Predicted Probability', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Distribution of Predicted Probabilities\n預測機率分布', fontsize=12)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Calibration Comparison\n模型校準比較', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('校準特性 Calibration characteristics:')
print('  Logistic Regression: 通常校準良好')
print('  Random Forest:      傾向保守，機率集中在 0.5 附近')
print('  Gradient Boosting:  中等校準品質')
print('  SVM:                需要 Platt Scaling')

---
## Part 6: 公平性指標 Fairness Metrics

公平性分析比較模型對不同**受保護群體 (Protected Groups)** 的表現差異。

三個核心指標：
- **Demographic Parity**: $P(\hat{Y}=1|A=a) = P(\hat{Y}=1|A=b)$
- **Equalized Odds**: $P(\hat{Y}=1|Y=y,A=a) = P(\hat{Y}=1|Y=y,A=b)$
- **Predictive Parity**: $P(Y=1|\hat{Y}=1,A=a) = P(Y=1|\hat{Y}=1,A=b)$

In [ ]:
# 計算公平性指標 Compute fairness metrics
mask_A = group_test == 'A'
mask_B = group_test == 'B'

def compute_group_metrics(y_true, y_pred, mask):
    """Compute metrics for a specific group."""
    y_t = y_true[mask]
    y_p = y_pred[mask]
    
    tp = np.sum((y_t == 1) & (y_p == 1))
    fp = np.sum((y_t == 0) & (y_p == 1))
    tn = np.sum((y_t == 0) & (y_p == 0))
    fn = np.sum((y_t == 1) & (y_p == 0))
    
    positive_rate = np.mean(y_p)  # P(Y_hat=1|A)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # True Positive Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Positive Predictive Value
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    
    return {
        'positive_rate': positive_rate,
        'tpr': tpr, 'fpr': fpr, 'ppv': ppv,
        'accuracy': accuracy,
        'base_rate': np.mean(y_t)
    }

metrics_A = compute_group_metrics(y_test, y_pred, mask_A)
metrics_B = compute_group_metrics(y_test, y_pred, mask_B)

# 計算公平性差距
dp_diff = abs(metrics_A['positive_rate'] - metrics_B['positive_rate'])
eo_tpr_diff = abs(metrics_A['tpr'] - metrics_B['tpr'])
eo_fpr_diff = abs(metrics_A['fpr'] - metrics_B['fpr'])
pp_diff = abs(metrics_A['ppv'] - metrics_B['ppv'])

# 視覺化
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 子圖 1: 各群體的關鍵指標比較
metric_names = ['Positive\nRate', 'TPR\n(Recall)', 'FPR', 'PPV\n(Precision)', 'Accuracy', 'Base\nRate']
vals_A = [metrics_A['positive_rate'], metrics_A['tpr'], metrics_A['fpr'],
          metrics_A['ppv'], metrics_A['accuracy'], metrics_A['base_rate']]
vals_B = [metrics_B['positive_rate'], metrics_B['tpr'], metrics_B['fpr'],
          metrics_B['ppv'], metrics_B['accuracy'], metrics_B['base_rate']]

x_pos = np.arange(len(metric_names))
width = 0.35
axes[0].bar(x_pos - width/2, vals_A, width, label='Group A', color='#3498DB', alpha=0.8)
axes[0].bar(x_pos + width/2, vals_B, width, label='Group B', color='#E74C3C', alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(metric_names, fontsize=9)
axes[0].set_ylabel('Value', fontsize=11)
axes[0].set_title('Metrics by Group\n各群體指標比較', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim(0, 1.1)

# 子圖 2: 公平性差距
fairness_metrics = ['Demographic\nParity Diff', 'Equal Opp\n(TPR Diff)',
                    'Equalized Odds\n(FPR Diff)', 'Predictive\nParity Diff']
diffs = [dp_diff, eo_tpr_diff, eo_fpr_diff, pp_diff]
bar_colors = ['red' if d > 0.1 else 'orange' if d > 0.05 else 'green' for d in diffs]

axes[1].barh(fairness_metrics, diffs, color=bar_colors, alpha=0.8)
axes[1].axvline(x=0.1, color='red', linestyle='--', linewidth=1.5, label='Unfair threshold (0.1)')
axes[1].axvline(x=0.05, color='orange', linestyle='--', linewidth=1.5, label='Warning threshold (0.05)')
axes[1].set_xlabel('Absolute Difference', fontsize=11)
axes[1].set_title('Fairness Disparity Metrics\n公平性差距指標', fontsize=12)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='x')

# 子圖 3: 分群混淆矩陣
ax = axes[2]
cm_A = confusion_matrix(y_test[mask_A], y_pred[mask_A])
cm_B = confusion_matrix(y_test[mask_B], y_pred[mask_B])

# 正規化為比例
cm_A_norm = cm_A.astype(float) / cm_A.sum()
cm_B_norm = cm_B.astype(float) / cm_B.sum()

categories = ['TN', 'FP', 'FN', 'TP']
vals_cm_A = [cm_A_norm[0,0], cm_A_norm[0,1], cm_A_norm[1,0], cm_A_norm[1,1]]
vals_cm_B = [cm_B_norm[0,0], cm_B_norm[0,1], cm_B_norm[1,0], cm_B_norm[1,1]]

x_cm = np.arange(4)
ax.bar(x_cm - width/2, vals_cm_A, width, label='Group A', color='#3498DB', alpha=0.8)
ax.bar(x_cm + width/2, vals_cm_B, width, label='Group B', color='#E74C3C', alpha=0.8)
ax.set_xticks(x_cm)
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylabel('Proportion', fontsize=11)
ax.set_title('Confusion Matrix by Group\n分群混淆矩陣比例', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Fairness Analysis: Group A vs Group B\n公平性分析', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('=== Fairness Summary ===')
print(f'Demographic Parity Difference:  {dp_diff:.4f} {"(UNFAIR)" if dp_diff > 0.1 else "(OK)"}')
print(f'Equal Opportunity (TPR) Diff:   {eo_tpr_diff:.4f} {"(UNFAIR)" if eo_tpr_diff > 0.1 else "(OK)"}')
print(f'Equalized Odds (FPR) Diff:      {eo_fpr_diff:.4f} {"(UNFAIR)" if eo_fpr_diff > 0.1 else "(OK)"}')
print(f'Predictive Parity Diff:         {pp_diff:.4f} {"(UNFAIR)" if pp_diff > 0.1 else "(OK)"}')

---
## Part 7: 資料偏誤 Bias in Data

訓練資料的不平衡或偏差是模型偏誤的最常見來源。
讓我們展示：當訓練資料中某個群體的正類比例人為偏低時，模型會產生什麼偏見。

In [ ]:
# 展示資料偏誤如何導致模型偏見
np.random.seed(42)

# 建立一個偏誤的訓練集：人為降低 Group B 的正類樣本
mask_train_A = group_train == 'A'
mask_train_B = group_train == 'B'

# 原始訓練集
print('=== Original Training Data ===')
print(f'Group A: {mask_train_A.sum()} samples, positive rate = {y_train[mask_train_A].mean():.3f}')
print(f'Group B: {mask_train_B.sum()} samples, positive rate = {y_train[mask_train_B].mean():.3f}')

# 建立偏誤資料：移除 Group B 的部分正類樣本
B_positive_indices = np.where(mask_train_B & (y_train == 1))[0]
n_remove = int(len(B_positive_indices) * 0.6)  # 移除 60% 的 B 正類
np.random.shuffle(B_positive_indices)
indices_to_remove = B_positive_indices[:n_remove]

keep_mask = np.ones(len(y_train), dtype=bool)
keep_mask[indices_to_remove] = False

X_train_biased = X_train_s[keep_mask]
y_train_biased = y_train[keep_mask]
group_train_biased = group_train[keep_mask]

print(f'\n=== Biased Training Data (removed 60% of B positives) ===')
mask_biased_A = group_train_biased == 'A'
mask_biased_B = group_train_biased == 'B'
print(f'Group A: {mask_biased_A.sum()} samples, positive rate = {y_train_biased[mask_biased_A].mean():.3f}')
print(f'Group B: {mask_biased_B.sum()} samples, positive rate = {y_train_biased[mask_biased_B].mean():.3f}')

# 訓練偏誤模型
clf_biased = LogisticRegression(random_state=42, max_iter=1000)
clf_biased.fit(X_train_biased, y_train_biased)
y_pred_biased = clf_biased.predict(X_test_s)

# 計算偏誤模型的公平性
metrics_A_biased = compute_group_metrics(y_test, y_pred_biased, mask_A)
metrics_B_biased = compute_group_metrics(y_test, y_pred_biased, mask_B)

# 視覺化比較
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：訓練資料的正類比例
ax = axes[0]
groups_bar = ['Group A\n(Original)', 'Group B\n(Original)',
              'Group A\n(Biased)', 'Group B\n(Biased)']
positive_rates = [
    y_train[mask_train_A].mean(), y_train[mask_train_B].mean(),
    y_train_biased[mask_biased_A].mean(), y_train_biased[mask_biased_B].mean()
]
bar_colors_data = ['#3498DB', '#E74C3C', '#85C1E9', '#F1948A']
ax.bar(groups_bar, positive_rates, color=bar_colors_data, alpha=0.8)
ax.set_ylabel('Positive Rate in Training Data', fontsize=11)
ax.set_title('Training Data Positive Rates\n訓練資料正類比例', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(positive_rates):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# 右圖：模型在測試集上的表現比較
ax = axes[1]
metric_compare = ['Positive\nRate', 'TPR', 'FPR', 'Accuracy']

# 原始模型
orig_vals = [metrics_A['positive_rate'] - metrics_B['positive_rate'],
             metrics_A['tpr'] - metrics_B['tpr'],
             metrics_A['fpr'] - metrics_B['fpr'],
             metrics_A['accuracy'] - metrics_B['accuracy']]

# 偏誤模型
biased_vals = [metrics_A_biased['positive_rate'] - metrics_B_biased['positive_rate'],
               metrics_A_biased['tpr'] - metrics_B_biased['tpr'],
               metrics_A_biased['fpr'] - metrics_B_biased['fpr'],
               metrics_A_biased['accuracy'] - metrics_B_biased['accuracy']]

x_bar = np.arange(len(metric_compare))
width = 0.35
ax.bar(x_bar - width/2, orig_vals, width, label='Original Model', color='steelblue', alpha=0.8)
ax.bar(x_bar + width/2, biased_vals, width, label='Biased Model', color='coral', alpha=0.8)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='Unfairness threshold')
ax.axhline(y=-0.1, color='red', linestyle='--', alpha=0.5)
ax.set_xticks(x_bar)
ax.set_xticklabels(metric_compare, fontsize=9)
ax.set_ylabel('Difference (Group A - Group B)', fontsize=10)
ax.set_title('Model Fairness Gap (A - B)\n模型公平性差距', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('How Data Bias Creates Model Bias\n資料偏誤如何產生模型偏見', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\n結論 Conclusion:')
print('  資料偏誤 (移除 Group B 正類) → 模型學到 Group B「更不可能」為正類')
print('  → 模型對 Group B 的正向預測率和 TPR 都更低 → 系統性歧視')

---
## Part 8: 偏誤緩解策略 Bias Mitigation Strategies

兩種常見的緩解方法：
1. **重新加權 (Reweighting)**：為弱勢群體的樣本賦予更高的訓練權重
2. **閾值調整 (Threshold Adjustment)**：為不同群體使用不同的決策閾值

In [ ]:
# 偏誤緩解策略 1: 重新加權 Reweighting
# 為 Group B 的正類樣本增加權重，補償被移除的部分
sample_weights = np.ones(len(y_train_biased))

# 計算每個群組/類別的比例，設定權重
for g in ['A', 'B']:
    for c in [0, 1]:
        mask_gc = (group_train_biased == g) & (y_train_biased == c)
        n_gc = mask_gc.sum()
        expected_prop = 0.25  # 每個群組/類別理想佔比
        actual_prop = n_gc / len(y_train_biased)
        if actual_prop > 0:
            sample_weights[mask_gc] = expected_prop / actual_prop

# 使用加權訓練
clf_reweight = LogisticRegression(random_state=42, max_iter=1000)
clf_reweight.fit(X_train_biased, y_train_biased, sample_weight=sample_weights)
y_pred_reweight = clf_reweight.predict(X_test_s)

# 偏誤緩解策略 2: 閾值調整 Threshold Adjustment
y_proba_biased = clf_biased.predict_proba(X_test_s)[:, 1]

# 找到使兩群體 FPR 相等的閾值
best_thresh_A = 0.5
best_thresh_B = 0.5
min_fpr_diff = float('inf')

for t_B in np.linspace(0.1, 0.9, 81):
    y_adj = y_pred_biased.copy()
    # 對 Group B 使用不同閾值
    y_adj[mask_B] = (y_proba_biased[mask_B] >= t_B).astype(int)
    y_adj[mask_A] = (y_proba_biased[mask_A] >= 0.5).astype(int)
    
    m_A = compute_group_metrics(y_test, y_adj, mask_A)
    m_B = compute_group_metrics(y_test, y_adj, mask_B)
    fpr_diff = abs(m_A['fpr'] - m_B['fpr'])
    
    if fpr_diff < min_fpr_diff:
        min_fpr_diff = fpr_diff
        best_thresh_B = t_B
        best_y_adj = y_adj.copy()

print(f'Threshold adjustment: Group A threshold = 0.50, Group B threshold = {best_thresh_B:.2f}')
print(f'Resulting FPR difference: {min_fpr_diff:.4f}')

# 比較三種方法
methods_results = {
    'Biased Model': y_pred_biased,
    'Reweighted Model': y_pred_reweight,
    'Threshold Adjusted': best_y_adj,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (method_name, y_p) in enumerate(methods_results.items()):
    m_A = compute_group_metrics(y_test, y_p, mask_A)
    m_B = compute_group_metrics(y_test, y_p, mask_B)
    
    ax = axes[idx]
    metrics_list = ['Pos Rate', 'TPR', 'FPR', 'PPV']
    vals_a = [m_A['positive_rate'], m_A['tpr'], m_A['fpr'], m_A['ppv']]
    vals_b = [m_B['positive_rate'], m_B['tpr'], m_B['fpr'], m_B['ppv']]
    
    x_idx = np.arange(len(metrics_list))
    ax.bar(x_idx - 0.17, vals_a, 0.34, label='Group A', color='#3498DB', alpha=0.8)
    ax.bar(x_idx + 0.17, vals_b, 0.34, label='Group B', color='#E74C3C', alpha=0.8)
    ax.set_xticks(x_idx)
    ax.set_xticklabels(metrics_list, fontsize=9)
    ax.set_ylabel('Value', fontsize=10)
    ax.set_title(method_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.1)
    
    dp = abs(m_A['positive_rate'] - m_B['positive_rate'])
    eo = abs(m_A['tpr'] - m_B['tpr'])
    ax.text(0.02, 0.02, f'DP Diff: {dp:.3f}\nEO Diff: {eo:.3f}',
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Bias Mitigation Strategies Comparison\n偏誤緩解策略比較', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\n緩解效果 Mitigation Effects:')
print('  Reweighting: 透過調整訓練權重平衡群體差異')
print('  Threshold Adjustment: 對弱勢群體使用較低閾值增加正向預測')
print('  兩者都能減少公平性差距，但可能犧牲整體準確率')

---
## Part 9: 模型穩健性 Model Robustness

穩健性測試：模型在面對**噪音資料**或**分布偏移 (Distribution Shift)** 時的表現。

我們對測試資料加入不同程度的噪音和偏移，觀察模型性能的下降曲線。

In [ ]:
# 模型穩健性測試 Robustness testing

# 測試模型 (使用原始模型)
models_robust = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# 訓練所有模型
for name, model in models_robust.items():
    model.fit(X_train_s, y_train)

# 測試不同噪音水準
noise_levels = np.linspace(0, 3.0, 20)
shift_levels = np.linspace(0, 3.0, 20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_rob = ['#E74C3C', '#3498DB', '#2ECC71']

# 左圖：高斯噪音 Gaussian Noise
ax = axes[0]
for (name, model), color in zip(models_robust.items(), colors_rob):
    accs_noise = []
    for noise in noise_levels:
        X_noisy = X_test_s + np.random.randn(*X_test_s.shape) * noise
        acc_n = model.score(X_noisy, y_test)
        accs_noise.append(acc_n)
    ax.plot(noise_levels, accs_noise, color=color, linewidth=2,
            label=name, marker='o', markersize=4)

ax.set_xlabel('Noise Level (std)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy vs Gaussian Noise\n準確率 vs 高斯噪音', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.4, 0.85)

# 右圖：均值偏移 Mean Shift
ax = axes[1]
for (name, model), color in zip(models_robust.items(), colors_rob):
    accs_shift = []
    for sh in shift_levels:
        X_shifted = X_test_s + sh  # 均值偏移
        acc_s = model.score(X_shifted, y_test)
        accs_shift.append(acc_s)
    ax.plot(shift_levels, accs_shift, color=color, linewidth=2,
            label=name, marker='s', markersize=4)

ax.set_xlabel('Mean Shift', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy vs Distribution Shift\n準確率 vs 分布偏移', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 0.85)

plt.suptitle('Model Robustness Under Data Perturbation\n模型在資料擾動下的穩健性',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('穩健性觀察 Robustness Observations:')
print('  - 所有模型在噪音/偏移增大時性能下降')
print('  - 線性模型 (LR) 對分布偏移可能更敏感')
print('  - 集成模型 (RF, GB) 通常有更好的穩健性')
print('  - 部署前必須測試模型對真實噪音的抵抗力')

---
## Exercises 練習題

完成以下練習來鞏固本週所學。

### Exercise 1: 找到使兩群體 FPR 相等的閾值

使用本週的資料集和基線模型 (`clf_base`)，
找到分別給 Group A 和 Group B 的最佳閾值，使得兩群體的 FPR 盡可能接近。

提示：
- 固定 Group A 的閾值為 0.5
- 搜尋 Group B 的閾值 (0.1 ~ 0.9)
- 計算每個閾值下兩群體的 FPR 差距

In [ ]:
# Exercise 1: 請在此撰寫你的程式碼
# TODO: Find threshold that equalizes FPR across groups

# y_proba_base = clf_base.predict_proba(X_test_s)[:, 1]
# for t_B in np.linspace(0.1, 0.9, 81):
#     y_adj = np.zeros_like(y_test)
#     y_adj[mask_A] = (y_proba_base[mask_A] >= 0.5).astype(int)
#     y_adj[mask_B] = (y_proba_base[mask_B] >= t_B).astype(int)
#     fpr_A = ...
#     fpr_B = ...
#     ...
# Print the best threshold and resulting FPR values

### Exercise 2: 比較不同分類器的公平性指標

訓練以下三個分類器，比較它們在 Demographic Parity 和 Equalized Odds 上的差異：
1. Logistic Regression
2. Random Forest
3. Gradient Boosting

哪個模型最「公平」？哪個模型最「準確」？

In [ ]:
# Exercise 2: 請在此撰寫你的程式碼
# TODO: Compare fairness metrics across different classifiers

# classifiers = {
#     'LR': LogisticRegression(random_state=42, max_iter=1000),
#     'RF': RandomForestClassifier(n_estimators=100, random_state=42),
#     'GB': GradientBoostingClassifier(n_estimators=100, random_state=42),
# }
# for name, clf in classifiers.items():
#     clf.fit(X_train_s, y_train)
#     y_p = clf.predict(X_test_s)
#     m_A = compute_group_metrics(y_test, y_p, mask_A)
#     m_B = compute_group_metrics(y_test, y_p, mask_B)
#     dp = abs(m_A['positive_rate'] - m_B['positive_rate'])
#     eo = abs(m_A['tpr'] - m_B['tpr'])
#     print(f'{name}: Accuracy={...}, DP Diff={dp:.4f}, EO Diff={eo:.4f}')

### Exercise 3: 實作重新加權以減少 Demographic Parity 差距

使用 `sample_weight` 參數訓練 `LogisticRegression`，
調整 Group A 和 Group B 的樣本權重，使得模型的 Demographic Parity Difference 低於 0.05。

提示：
- 增加 Group B 正類的權重
- 或降低 Group A 正類的權重
- 嘗試不同的權重比例

In [ ]:
# Exercise 3: 請在此撰寫你的程式碼
# TODO: Implement reweighting to reduce DP gap

# for w_B_pos in [1.0, 1.5, 2.0, 2.5, 3.0]:
#     weights = np.ones(len(y_train))
#     mask_B_pos = (group_train == 'B') & (y_train == 1)
#     weights[mask_B_pos] = w_B_pos
#     
#     clf_w = LogisticRegression(random_state=42, max_iter=1000)
#     clf_w.fit(X_train_s, y_train, sample_weight=weights)
#     y_p = clf_w.predict(X_test_s)
#     ...
#     print(f'Weight={w_B_pos}: DP Diff={dp:.4f}, Accuracy={acc:.4f}')

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **混淆矩陣深入**：在特徵空間中視覺化 TP/FP/TN/FN 的分布
2. **分類指標 vs 閾值**：Precision、Recall、F1、Specificity 隨閾值的變化
3. **多類別 ROC/PR**：One-vs-Rest 策略，每個類別的 AUC
4. **閾值優化**：根據業務場景選擇最佳閾值
5. **校準曲線**：比較不同模型的預測機率可靠性
6. **公平性指標**：Demographic Parity、Equalized Odds、Predictive Parity
7. **資料偏誤**：訓練資料不平衡如何導致系統性歧視
8. **偏誤緩解**：重新加權和閾值調整的效果
9. **穩健性測試**：噪音和分布偏移下的性能下降

### 關鍵收穫 Key Takeaways
- 閾值是**部署決策**，不是模型的一部分
- 準確率高不代表模型公平 — 必須分群體檢視
- 資料偏誤是模型偏見最常見的來源
- 公平性定義之間存在不可能定理 — 無法同時滿足所有定義
- 模型部署前必須進行穩健性測試

### 下週預告 Next Week Preview
**第 16 週：MLOps 簡介**
- 模型版本管理
- 模型部署流程
- 持續監控與維護